In [2]:
import os
os.environ["USE_TF"]="0"
from sklearn.metrics import label_ranking_average_precision_score
from transformers import AutoTokenizer
import torch
from torch import nn
import pandas as pd
import tokenizer
import sklearn
import numpy as np
from torch.utils.data import DataLoader,Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoModelForSequenceClassification
from transformers import Trainer,TrainingArguments

C:\Users\LOQ\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
x=pd.read_csv("IMDB Dataset.csv")
x.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
x=x.iloc[:10000]
x.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
x1=x["review"]
y=x["sentiment"]

In [6]:
from sklearn.preprocessing import LabelEncoder
lbl=LabelEncoder()
y=lbl.fit_transform(y)
y

array([1, 1, 1, ..., 0, 0, 1], shape=(10000,))

In [7]:
tk=AutoTokenizer.from_pretrained("bert-base-uncased")

In [8]:
tk2=tk(x1.tolist(),padding=True,
       max_length=126,
       truncation=True,)

In [9]:
print(list(tk2.keys()))

['input_ids', 'token_type_ids', 'attention_mask']


In [10]:
print(tk2["input_ids"][:10])

[[101, 2028, 1997, 1996, 2060, 15814, 2038, 3855, 2008, 2044, 3666, 2074, 1015, 11472, 2792, 2017, 1005, 2222, 2022, 13322, 1012, 2027, 2024, 2157, 1010, 2004, 2023, 2003, 3599, 2054, 3047, 2007, 2033, 1012, 1026, 7987, 1013, 1028, 1026, 7987, 1013, 1028, 1996, 2034, 2518, 2008, 4930, 2033, 2055, 11472, 2001, 2049, 24083, 1998, 4895, 10258, 2378, 8450, 5019, 1997, 4808, 1010, 2029, 2275, 1999, 2157, 2013, 1996, 2773, 2175, 1012, 3404, 2033, 1010, 2023, 2003, 2025, 1037, 2265, 2005, 1996, 8143, 18627, 2030, 5199, 3593, 1012, 2023, 2265, 8005, 2053, 17957, 2007, 12362, 2000, 5850, 1010, 3348, 2030, 4808, 1012, 2049, 2003, 13076, 1010, 1999, 1996, 4438, 2224, 1997, 1996, 2773, 1012, 1026, 7987, 1013, 1028, 1026, 7987, 1013, 1028, 2009, 2003, 2170, 11472, 102], [101, 1037, 6919, 2210, 2537, 1012, 1026, 7987, 1013, 1028, 1026, 7987, 1013, 1028, 1996, 7467, 6028, 2003, 2200, 14477, 4757, 24270, 1011, 2200, 2214, 1011, 2051, 1011, 4035, 4827, 1998, 3957, 1037, 16334, 1010, 1998, 2823, 17964, 

In [11]:
x_train,x_temp,y_train,y_temp=train_test_split(x1,y,test_size=0.30, random_state=42,stratify=y)
x_val,x_test,y_val,y_test=train_test_split(x_temp,y_temp,test_size=0.5,random_state=42,stratify=y_temp)

print('train:',len(x_train),'val:',len(x_val),'test:',len(x_test))

train: 7000 val: 1500 test: 1500


In [12]:
model=AutoModelForSequenceClassification.from_pretrained("bert-base-uncased",num_labels=2,cache_dir="./bert_model")
tk_train=tk(x_train.tolist(),padding=True, truncation=True, max_length=126)
tk_val=tk(x_val.tolist(),padding=True, truncation=True, max_length=126)
tk_test=tk(x_test.tolist(),padding=True, truncation=True, max_length=126)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
class SentimentDataset(Dataset):
    def __init__(self,encoding,labels):
        self.encoding=encoding
        self.labels=labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self,idx):
         itm={key:torch.tensor(val[idx])for key,val in self.encoding.items()}
         itm["labels"]=torch.tensor(self.labels[idx])
         return itm

train_s=SentimentDataset(tk_train,y_train.tolist())
val_s=SentimentDataset(tk_val,y_val.tolist())
test_s=SentimentDataset(tk_test,y_test.tolist())

In [15]:
dev=torch.device("cuda")
model=model.to(dev)

ld=TrainingArguments(
    output_dir="./bert_model",
    num_train_epochs=1,
    gradient_accumulation_steps=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    save_strategy='epoch',
    eval_strategy='epoch',
    logging_steps=30,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    learning_rate=2e-5,
    weight_decay=0.01,
    save_total_limit=1,
    fp16=True,

)
tra=Trainer(
    model=model,
    args=ld,
    train_dataset=train_s,
    eval_dataset=val_s,
)

In [16]:
tra.train()

Epoch,Training Loss,Validation Loss
1,0.386651,0.318092


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.90it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber

TrainOutput(global_step=1000, training_loss=0.38363722944259643, metrics={'train_runtime': 149.065, 'train_samples_per_second': 53.668, 'train_steps_per_second': 6.708, 'total_flos': 517999890240000.0, 'train_loss': 0.38363722944259643, 'epoch': 1.0})

In [18]:
tra.save_model("./english_sentiment_analysis")
tk.save_pretrained("./english_sentiment_analysis")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.38it/s]


('./english_sentiment_analysis\\tokenizer_config.json',
 './english_sentiment_analysis\\tokenizer.json')

In [23]:
txt="yess,My mood is really good this song is really amazing"
inp=tk(txt,return_tensors="pt",truncation=True,max_length=126).to(dev)
model.eval()
with torch.no_grad():
    out=model(**inp)
    pr=torch.argmax(out.logits,dim=1).item()
lbl={1:"positive",0:"negative"}
print(lbl[pr])

positive


In [24]:
txt2="This is really bad it makes me feel bad."
inp2=tk(txt2,return_tensors="pt",truncation=True,max_length=126).to(dev)
model.eval()
with torch.no_grad():
    out2=model(**inp2)
    pr2=torch.argmax(out2.logits,dim=1).item()
lbl2={1:"positive",0:"negative"}
print(lbl2[pr2])

negative


In [25]:
txt3="I don't feel good about this matter; it really upset me."
inp3=tk(txt3,return_tensors="pt",truncation=True,max_length=126).to(dev)
model.eval()
with torch.no_grad():
    out3=model(**inp3)
    pr3=torch.argmax(out3.logits,dim=1).item()
lbl3={1:"positive",0:"negative"}
print(lbl3[pr3])

negative


In [27]:
txt4="This is amazing!!!"
inp4=tk(txt4,return_tensors="pt",truncation=True,max_length=126).to(dev)
model.eval()
with torch.no_grad():
    out4=model(**inp4)
    pr4=torch.argmax(out4.logits,dim=1).item()
lbl4={1:"positive",0:"negative"}
print(lbl4[pr4])

positive


In [34]:
from sklearn.metrics import classification_report
pred_A=[]
model.eval()
with torch.no_grad():
    for batch  in DataLoader(test_s, batch_size=16, shuffle=False):
        inpu={k:v.to(dev) for k,v in batch.items() if k!='labels'}
        outp=model(**inpu)
        pred_i=torch.argmax(outp.logits,dim=1)
        pred_A.extend(pred_i.cpu().tolist())
print(classification_report(y_test.tolist(),pred_A,target_names=["positive","negative"]))

              precision    recall  f1-score   support

    positive       0.90      0.85      0.88       996
    negative       0.86      0.91      0.88      1004

    accuracy                           0.88      2000
   macro avg       0.88      0.88      0.88      2000
weighted avg       0.88      0.88      0.88      2000

